In [ ]:
%sql
-- fact generation (date and hour keys in Brussels local time, ts_utc stays the technical key)
USE CATALOG energy;
SET TIME ZONE 'UTC';

INSERT OVERWRITE d1_gold.fact_generation_hourly
SELECT
  g.ts_utc,
  from_utc_timestamp(g.ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(g.ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(g.ts_utc, 'Europe/Brussels')) AS hour_key,
  g.zone_code,
  g.psr_code,
  GREATEST(g.generation_mwh, 0) AS generation_mwh,
  GREATEST(g.generation_mwh, 0) * p.ef_lifecycle_g_kwh / 1000 AS emissions_lca_t,   -- MWh x g/kWh / 1000 = t
  GREATEST(g.generation_mwh, 0) * p.ef_direct_g_kwh / 1000 AS emissions_direct_t
FROM d1_silver.silver_generation_hourly g
JOIN d1_gold.dim_production_type p
  ON g.psr_code = p.psr_code;

In [ ]:
%sql
-- generation, emissions and renewable share per zone and hour (input of the flow tracing)
CREATE OR REPLACE TEMPORARY VIEW zone_hour_balance AS
SELECT
  f.zone_code,
  f.ts_utc,
  SUM(f.generation_mwh) AS production_mwh,
  SUM(f.emissions_lca_t) AS emissions_lca_t,
  SUM(f.emissions_direct_t) AS emissions_direct_t,
  SUM(CASE WHEN p.is_renewable THEN f.generation_mwh ELSE 0 END) AS renewable_mwh,
  SUM(CASE WHEN p.is_low_carbon THEN f.generation_mwh ELSE 0 END) AS low_carbon_mwh
FROM d1_gold.fact_generation_hourly f
JOIN d1_gold.dim_production_type p
  ON f.psr_code = p.psr_code
GROUP BY f.zone_code, f.ts_utc;

In [ ]:
# FLOW TRACING
# for each hour, solve the intensity c of the electricity mix of the modeled zones:
#   c_n * (P_n + imports into n) = E_n + sum over origins k of (c_k * F_k->n)
# c_k is unknown for a modeled origin, and equal to the production intensity (or the fallback) for an external one
import numpy as np
import pandas as pd

spark.conf.set("spark.sql.session.timeZone", "UTC")

zones = spark.table("d1_gold.dim_zone").toPandas().set_index("zone_code")
modeled = zones.index[zones.is_modeled].tolist()

balance = spark.table("zone_hour_balance").toPandas()
flows = spark.table("d1_silver.silver_flows_hourly").where("flow_mwh > 0").toPandas()
flows_by_hour = dict(tuple(flows.groupby("ts_utc")))
no_flows = flows.iloc[0:0]

results = []
for ts, hour in balance.groupby("ts_utc"):
    hour = hour.set_index("zone_code")
    if not all(z in hour.index for z in modeled):
        continue    # incomplete hour: not traced

    A = np.diag([float(hour.at[z, "production_mwh"]) for z in modeled])
    b = hour.loc[modeled, ["emissions_lca_t", "emissions_direct_t"]].to_numpy(dtype=float)

    for f in flows_by_hour.get(ts, no_flows).itertuples():
        if f.to_zone not in modeled:
            continue
        i = modeled.index(f.to_zone)
        A[i, i] += f.flow_mwh
        if f.from_zone in modeled:
            A[i, modeled.index(f.from_zone)] -= f.flow_mwh
        elif f.from_zone in hour.index and hour.at[f.from_zone, "production_mwh"] > 0:
            origin = hour.loc[f.from_zone]
            b[i] += f.flow_mwh * np.array([origin.emissions_lca_t, origin.emissions_direct_t]) / origin.production_mwh
        else:
            fallback = zones.loc[f.from_zone]   # e.g. GB: generation not published on ENTSO-E
            b[i] += f.flow_mwh * np.array([fallback.fallback_ci_lca_g_kwh, fallback.fallback_ci_direct_g_kwh]) / 1000

    if (np.diag(A) <= 0).any():
        continue    # a zone with neither generation nor imports: not traced

    c = np.linalg.solve(A, b)    # t/MWh
    for i, z in enumerate(modeled):
        results.append((ts, z, c[i, 0] * 1000, c[i, 1] * 1000))

traced = pd.DataFrame(results, columns=["ts_utc", "zone_code", "ci_mix_lca_g_kwh", "ci_mix_direct_g_kwh"])
spark.createDataFrame(traced).createOrReplaceTempView("flow_tracing_results")
print(len(traced), "zone-hours traced out of", len(balance[balance.zone_code.isin(modeled)]))

In [ ]:
%sql
-- intensity of each flow origin: solved mix for modeled zones, production intensity for the others, fallback if missing
CREATE OR REPLACE TEMPORARY VIEW origin_intensity AS
SELECT
  f.from_zone,
  f.to_zone,
  f.ts_utc,
  f.flow_mwh,
  COALESCE(r.ci_mix_lca_g_kwh, b.emissions_lca_t * 1000 / NULLIF(b.production_mwh, 0), z.fallback_ci_lca_g_kwh) AS ci_lca_g_kwh,
  COALESCE(r.ci_mix_direct_g_kwh, b.emissions_direct_t * 1000 / NULLIF(b.production_mwh, 0), z.fallback_ci_direct_g_kwh) AS ci_direct_g_kwh,
  r.zone_code IS NULL AND COALESCE(b.production_mwh, 0) = 0 AS is_fallback
FROM d1_silver.silver_flows_hourly f
LEFT JOIN flow_tracing_results r
  ON f.from_zone = r.zone_code AND f.ts_utc = r.ts_utc
LEFT JOIN zone_hour_balance b
  ON f.from_zone = b.zone_code AND f.ts_utc = b.ts_utc
LEFT JOIN d1_gold.dim_zone z
  ON f.from_zone = z.zone_code;

In [ ]:
%sql
-- fact flow (CO2 embedded in each flow)
INSERT OVERWRITE d1_gold.fact_flow_hourly
SELECT
  ts_utc,
  from_utc_timestamp(ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(ts_utc, 'Europe/Brussels')) AS hour_key,
  from_zone AS from_zone_code,
  to_zone AS to_zone_code,
  flow_mwh,
  flow_mwh * ci_lca_g_kwh / 1000 AS embedded_emissions_lca_t,
  flow_mwh * ci_direct_g_kwh / 1000 AS embedded_emissions_direct_t
FROM origin_intensity;

In [ ]:
%sql
-- fact carbon (one row per modeled zone and hour)
INSERT OVERWRITE d1_gold.fact_carbon_hourly
WITH imports AS (
  SELECT
    to_zone AS zone_code,
    ts_utc,
    SUM(flow_mwh) AS imports_mwh,
    SUM(flow_mwh * ci_lca_g_kwh / 1000) AS imported_emissions_lca_t,
    SUM(flow_mwh * ci_direct_g_kwh / 1000) AS imported_emissions_direct_t,
    SUM(CASE WHEN is_fallback THEN flow_mwh ELSE 0 END) AS imports_fallback_ci_mwh
  FROM origin_intensity
  GROUP BY to_zone, ts_utc
),
exports AS (
  SELECT from_zone AS zone_code, ts_utc, SUM(flow_mwh) AS exports_mwh
  FROM origin_intensity
  GROUP BY from_zone, ts_utc
)
SELECT
  b.ts_utc,
  from_utc_timestamp(b.ts_utc, 'Europe/Brussels') AS datetime_local,
  CAST(date_format(from_utc_timestamp(b.ts_utc, 'Europe/Brussels'), 'yyyyMMdd') AS INT) AS date_key,
  hour(from_utc_timestamp(b.ts_utc, 'Europe/Brussels')) AS hour_key,
  b.zone_code,
  b.production_mwh,
  l.load_mwh,
  COALESCE(i.imports_mwh, 0) AS imports_mwh,
  COALESCE(e.exports_mwh, 0) AS exports_mwh,
  COALESCE(i.imports_mwh, 0) - COALESCE(e.exports_mwh, 0) AS net_imports_mwh,
  b.renewable_mwh,
  b.low_carbon_mwh,
  b.emissions_lca_t AS emissions_production_lca_t,
  b.emissions_direct_t AS emissions_production_direct_t,
  r.ci_mix_lca_g_kwh * l.load_mwh / 1000 AS emissions_consumption_lca_t,
  r.ci_mix_direct_g_kwh * l.load_mwh / 1000 AS emissions_consumption_direct_t,
  i.imported_emissions_lca_t,
  i.imported_emissions_direct_t,
  b.emissions_lca_t * 1000 / NULLIF(b.production_mwh, 0) AS ci_production_lca_g_kwh,
  r.ci_mix_lca_g_kwh AS ci_consumption_lca_g_kwh,
  b.emissions_direct_t * 1000 / NULLIF(b.production_mwh, 0) AS ci_production_direct_g_kwh,
  r.ci_mix_direct_g_kwh AS ci_consumption_direct_g_kwh,
  COALESCE(i.imports_fallback_ci_mwh, 0) AS imports_fallback_ci_mwh,
  r.zone_code IS NOT NULL AS is_flow_traced
FROM zone_hour_balance b
JOIN d1_gold.dim_zone z
  ON b.zone_code = z.zone_code AND z.is_modeled
LEFT JOIN d1_silver.silver_load_hourly l
  ON b.zone_code = l.zone_code AND b.ts_utc = l.ts_utc
LEFT JOIN flow_tracing_results r
  ON b.zone_code = r.zone_code AND b.ts_utc = r.ts_utc
LEFT JOIN imports i
  ON b.zone_code = i.zone_code AND b.ts_utc = i.ts_utc
LEFT JOIN exports e
  ON b.zone_code = e.zone_code AND b.ts_utc = e.ts_utc;